# 02. Grad-CAM: ResNet50 và ViT-B/16

So sánh Grad-CAM theo **logit lớp ground-truth**, kể cả dự đoán sai.
ResNet dùng `layer4[-1]`, ViT dùng `encoder.layers[11].ln_1` trước self-attention cuối.
Giữ weights ResNet50 IMAGENET1K_V2 và ViT-B/16 IMAGENET1K_V1 cùng preprocessing chính thức.
Hai crop khác nhau được căn chỉnh trên vùng giao trong tọa độ ảnh gốc.
Lưới chính 7×7, kiểm tra độ nhạy 14×14; metrics: Spearman, Top-20% IoU và entropy chuẩn hóa.

Notebook chạy độc lập: setup → chọn ảnh → chạy/resume → phân tích → export.
`experiments/gradcam/`: `methods.py` (heatmap), `experiment.py` (cohort/checkpoint),
`analysis.py` (metrics/bảng/hình/export). Tái sử dụng `src` và giữ nguyên các module chung.

## 1. Environment and synchronization

Colab: open this notebook from the personal public repository and select a GPU runtime.
Run the setup cell to clone/pull `feat/gradcam`, mount Drive, prepare data and install dependencies.
No GitHub token is needed. On Windows, run `.\sync.ps1` after editing, then rerun setup here.

Input: `MyDrive/cv_2026/data_10k.zip` containing 14 Parquet shards under `data_10k/`.
Setup extracts the shards to `/content` once per runtime and verifies the manifest and IDs/labels.
Code always comes from Git; set `DATA_DIR` only to override the extracted dataset.
Results/checkpoints stay under `MyDrive/cv_2026/metrics/` and `MyDrive/cv_2026/figures/`.
Weights are cached under `MyDrive/cv_2026/torch-cache/`.

Stop an active run before pulling. When available, autoreload refreshes Python modules before
subsequent cells; rerun configuration and experiment cells to recreate objects. If the runtime
cannot load autoreload, setup continues; restart the Python session after code changes. Restart the runtime session when
changing model classes or dependencies, or if an old object still behaves unexpectedly.
Git pull updates Python files, not the notebook cells already open in the Colab editor.
Reopen this notebook from GitHub when its cells change.

Local: use the local repository and dataset; no Drive mount or automatic dependency installation.
See [workflow guide](../docs/personal_remote_colab.md).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import runpy

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab defaults are ready to use. Local execution keeps the local repository.
PROJECT_DIR = None
DATA_DIR = None
RESULTS_DIR = None
PROJECT_ZIP = "cv_2026/data_10k.zip"  # Input ZIP relative to MyDrive.
MOUNT_DRIVE = True
INSTALL_DEPENDENCIES = IN_COLAB
REPO_URL = "https://github.com/thuhang012/cnn-vit-interpretability-personal.git"
BRANCH = "feat/gradcam"

if IN_COLAB:
    checkout = Path(PROJECT_DIR or "/content/cnn-vit-interpretability").resolve()

    def git(*args):
        return subprocess.check_output(["git", *args], text=True).strip()

    if (checkout / ".git").is_dir():
        if git("-C", str(checkout), "remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("Checkout points to a different repository.")
        if git("-C", str(checkout), "branch", "--show-current") != BRANCH:
            raise RuntimeError("Checkout is on a different branch.")
        if git("-C", str(checkout), "status", "--porcelain"):
            raise RuntimeError("Save Colab checkout edits separately before pulling.")
        git("-C", str(checkout), "pull", "--ff-only", "origin", BRANCH)
    else:
        git("clone", "--branch", BRANCH, REPO_URL, str(checkout))
    print("Code commit:", git("-C", str(checkout), "rev-parse", "--short", "HEAD"))
    configure = runpy.run_path(str(checkout / "scripts/colab_setup.py"))["setup"]
    ROOT, DATA_DIR, RESULTS_DIR = configure(
        checkout, data_dir=DATA_DIR, results_dir=RESULTS_DIR,
        project_zip=PROJECT_ZIP, mount_drive=MOUNT_DRIVE,
        install_dependencies=INSTALL_DEPENDENCIES,
    )
else:
    default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    ROOT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else default_root.resolve()

    def project_path(value, default):
        path = Path(value).expanduser() if value is not None else ROOT / default
        return (path if path.is_absolute() else ROOT / path).resolve()

    DATA_DIR = project_path(DATA_DIR, "data/imagenet/data")
    RESULTS_DIR = project_path(RESULTS_DIR, "results")
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                        str(ROOT / "requirements.txt")], check=True)

if not (ROOT / "configs/analysis_set.json").is_file():
    raise FileNotFoundError("PROJECT_DIR must point to the repository root.")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder is missing: {DATA_DIR}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload edited Python modules on subsequent cells after pulling new code.
try:
    shell = get_ipython()
except NameError:
    shell = None
AUTORELOAD_AVAILABLE = False
if shell is not None:
    try:
        shell.run_line_magic("load_ext", "autoreload")
        shell.run_line_magic("autoreload", "2")
        AUTORELOAD_AVAILABLE = True
    except ImportError as error:
        print(f"Optional autoreload is unavailable ({error}). Setup can continue.")
        print("After pulling Python code changes, restart the Python session and rerun cells to load the new code.")
print(f"Environment: {'Colab' if IN_COLAB else 'local'} | Project: {ROOT}")
print(f"Data: {DATA_DIR}\nResults/checkpoints: {RESULTS_DIR}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from experiments.gradcam.experiment import prepare_cohorts, experiment_stores
from experiments.gradcam.analysis import analyze, show_results, plot_summary, plot_examples, export_final

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

In [ ]:
RUN_EXPERIMENT = True      # False: only read existing checkpoints.
SAVE_FINAL = True          # False: inspect/replot without writing final files.
CHECK_PINNED_DATA = False  # Optional full source/hash verification.
ANALYSIS_SIZE = 10000      # 5000: class-balanced trial; 10000: shared cohort.
SEED = 42
BATCH_SIZE = 1             # Explicit batch; no pilot or automatic batch selection.
ALLOW_DOWNLOAD = IN_COLAB    # Opt in if pretrained weights are missing.
RUN_TAG = f"gradcam_v1_n{ANALYSIS_SIZE}"
CONDITION = "E1"

METRICS_ROOT = RESULTS_DIR / "metrics/experiments/gradcam" / RUN_TAG
OUT = METRICS_ROOT / "intermediate"
FINAL = METRICS_ROOT / "final"
FIG = RESULTS_DIR / "figures/experiments/gradcam" / RUN_TAG / "final"
stores = result = None
figures = {}

## 2. Manifest và cohort

Giữ nguyên manifest 10.000 ảnh (10 ảnh/lớp), seed và cách ghép dự đoán baseline theo ID.
Nhóm đúng/sai vẫn dựa trên baseline của cả ResNet và ViT; không cần chạy lại baseline.
Grad-CAM dùng 10.000 ảnh mặc định; tùy chọn 5.000 ảnh cân bằng lớp chỉ là lượt chạy thử.

In [ ]:
if CHECK_PINNED_DATA:
    # The existing checker uses the dataset path in configs/analysis_set.json.
    if DATA_DIR != (ROOT / "data/imagenet/data").resolve():
        raise ValueError("CHECK_PINNED_DATA uses the configured dataset; run it with the default DATA_DIR.")
    subprocess.run([sys.executable, str(ROOT / "src/create_analysis_manifest.py"), "--check"],
                   cwd=ROOT, check=True)
selection = prepare_cohorts(ROOT, ANALYSIS_SIZE, SEED)
display(selection.groupby("group").size().rename("n_images").to_frame())
print(f"{CONDITION}: {len(selection)} images | seed={SEED}")

## 3. Chạy hoặc tiếp tục checkpoint

Không yêu cầu pilot. Batch mặc định 1; tăng `BATCH_SIZE` khi phù hợp bộ nhớ.
Các kiểm tra logits, heatmap và hook vẫn được giữ trong lúc trích xuất.
ID đã commit không chạy lại; checkpoint khác cấu hình bị từ chối.
OOM dừng lượt chạy, không tự giảm cohort. Chọn run ID mới khi thay code/cấu hình/batch.
Để xem lại, đặt `RUN_EXPERIMENT=False` và `SAVE_FINAL=False`, giữ cùng run ID.

In [ ]:
if not isinstance(BATCH_SIZE, int) or isinstance(BATCH_SIZE, bool) or BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be a positive integer")
batch_sizes = {name: BATCH_SIZE for name in ("resnet50", "vit_b_16")} if RUN_EXPERIMENT else None
stores = experiment_stores(ROOT, selection, CONDITION, OUT, batch_sizes=batch_sizes,
                           device=DEVICE, seed=SEED, allow_download=ALLOW_DOWNLOAD,
                           data_dir=DATA_DIR)
result = analyze(*stores) if stores is not None else None

## 4. Chỉ số và độ bao phủ

Bảng tổng hợp giữ nguyên nội dung: chỉ số chính, nhóm đúng/sai và coverage.
Heatmap lỗi/suy biến được gắn cờ; metric không xác định giữ NaN, không thay bằng 0.
Không hiển thị bảng per-image hoặc danh sách ID đầy đủ; các bảng này có trong file export.

In [ ]:
if result is not None:
    show_results({CONDITION: result})
else:
    print(f"{CONDITION} chưa hoàn tất; xem tiến độ ở phần trên.")

## 5. Visualization

Giữ nguyên biểu đồ định lượng, cách chọn ảnh theo nhóm baseline và các panel heatmap.
Mỗi input/overlay dùng cùng crop chính thức; tiêu đề ghi rõ model và phương pháp.

In [ ]:
figures = {}
if result is not None:
    figures = {f"{CONDITION}_metrics": plot_summary(result, CONDITION),
               f"{CONDITION}_examples": plot_examples(ROOT, stores, selection, data_dir=DATA_DIR)}
    plt.show()

## 6. Export riêng cho E1

Không cần chạy notebook còn lại. Lưu `metrics.csv`, `per_image.csv`,
CSV dự đoán của hai model và `metadata.json` vào `FINAL`;
hai hình `E1_metrics.png`, `E1_examples.png` vào `FIG`.
Giữ schema số liệu cũ; tên thí nghiệm trong metadata/CSV là `gradcam`.

Accuracy dùng dự đoán trích xuất thành công, kèm số ảnh yêu cầu/lỗi; đây là kiểm tra
tính nhất quán dự đoán, không phải thước đo chất lượng heatmap.
Không ghi đè final hoặc chuyển checkpoint cũ từ notebook chung.
Trên Colab, checkpoint và final đều được lưu trên Drive; khi ngắt phiên có thể mất
phần batch chưa commit, nhưng lượt chạy sau tiếp tục từ các chunk đã lưu.
Notebook được lưu không kèm output nhúng.

In [ ]:
if SAVE_FINAL and result is not None:
    accuracy = export_final(stores, {CONDITION: result}, figures, FINAL, FIG)
    print("Prediction accuracy: percentages on successful extractions; failures listed separately")
    display(accuracy.round(4))
elif not SAVE_FINAL:
    print("Preview only: no final files written.")
else:
    print(f"Final export skipped: {CONDITION} is incomplete.")